# Phishing-Email SLM Fine-Tuning Starter (QLoRA) — Colab-Safe Version
**Project:** Small but Sufficient? — SLM vs. LLM phishing-email detection

This notebook covers **Weeks 5–6** of your project plan: naive LoRA fine-tuning and CoT-distilled LoRA fine-tuning of a small model on phishing-email classification, with checkpoints saved to **Google Drive** so a Colab disconnect never loses your trained adapter.

**How to use, step by step:**
1. Open this file in Google Colab (Colab → File → Upload notebook, or open from GitHub if you pushed the repo).
2. Runtime → Change runtime type → **T4 GPU** (free tier is enough).
3. Run Cell 1 (installs) then Cell 2 (Drive mount) — you'll get a Google auth popup, approve it.
4. Edit **Cell 4 (Config)** — this is the only cell you should need to change between runs.
5. Run the rest top to bottom. It works today on 3 synthetic placeholder rows so you can confirm the whole pipeline before your real Week 2 dataset exists.
6. Run it **twice per model**: once with `USE_COT = False` (Week 5), once with `USE_COT = True` (Week 6) — everything else stays identical so the comparison is fair.


In [ ]:
# CELL 1 — Install pinned dependencies
# Pinned versions matching requirements.txt for reproducible TRL 0.15+ compatibility
!pip install -q transformers==4.49.0 peft==0.14.0 trl==0.15.2 bitsandbytes==0.45.3 accelerate==1.4.0 datasets>=3.0.0 pandas scikit-learn
!pip uninstall -y wandb -q


In [ ]:
# CELL 2 — Mount Google Drive (Colab-safe checkpoint storage)
import os
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/phishing-slm-research"
    os.makedirs(f"{DRIVE_ROOT}/data/processed", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/finetuning/checkpoints", exist_ok=True)
    print("Mounted Google Drive. Project root:", DRIVE_ROOT)
else:
    DRIVE_ROOT = "."
    print("Local environment root:", DRIVE_ROOT)


In [ ]:
# CELL 3 — Imports and GPU check
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device: ", torch.cuda.get_device_name(0))
    print("GPU Memory: ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU.")


In [ ]:
# CELL 4 — Config (Arm A Naive QLoRA)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
USE_COT = False       # Arm A: False = naive direct classification
SMOKE_TEST = False    # Set True for Step 2 smoke test (200 rows, ~40 steps)
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

# Hardware precision decision for T4 GPU: fp16 is native and optimal on Turing
USE_FP16 = True
USE_BF16 = False

DATA_PATH = f"{DRIVE_ROOT}/data/processed/phishing_train.csv"
OUTPUT_DIR = f"{DRIVE_ROOT}/finetuning/checkpoints/{MODEL_NAME.split('/')[-1]}-{'cot' if USE_COT else 'naive'}"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS = 3
MAX_BODY_TOKENS = 800  # Pre-format body token budget to prevent target clipping
SAVE_STEPS = 200       # Saves checkpoint every ~20-30 mins to protect against disconnects

print("Model:         ", MODEL_NAME)
print("Mode:          ", 'CoT-distilled' if USE_COT else 'Naive label-only (Arm A)')
print("Smoke test:    ", SMOKE_TEST)
print("Data path:     ", DATA_PATH)
print("Output dir:    ", OUTPUT_DIR)


In [ ]:
# CELL 5 — Load data (phishing_train.csv only — Auto-generates to Drive if missing)
import os, hashlib

if not os.path.exists(DATA_PATH):
    print(f"Notice: phishing_train.csv not found at {DATA_PATH}.")
    print("Auto-curating authoritative splits from Hugging Face into Drive (takes ~30s)...")
    from datasets import load_dataset
    from sklearn.model_selection import train_test_split
    ds = load_dataset("puyang2025/seven-phishing-email-datasets")
    df_all = ds["train"].to_pandas()
    sources = ["Enron", "CEAS-08", "Assassin", "TREC-05", "TREC-06", "TREC-07"]
    df_clean = df_all[df_all["dataset_name"].isin(sources)].copy().dropna(subset=["text", "label"])
    df_clean = df_clean[df_clean["text"].str.strip().str.len() > 0]
    df_clean = df_clean.groupby("dataset_name", group_keys=False).apply(lambda g: g.sample(n=min(len(g), 5000), random_state=42)).reset_index(drop=True)
    df_clean["body_hash"] = df_clean["text"].apply(lambda t: hashlib.md5(t.strip().lower().encode()).hexdigest())
    df_clean = df_clean.drop_duplicates(subset="body_hash").drop(columns="body_hash").rename(columns={"text": "body"})
    df_clean["subject"] = df_clean["subject"].fillna("")
    df_clean["email_id"] = range(len(df_clean))
    df_clean["strat_key"] = df_clean["label"].astype(str) + "_" + df_clean["dataset_name"]
    train_df, temp_df = train_test_split(df_clean, test_size=0.30, stratify=df_clean["strat_key"], random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=(0.20/0.30), stratify=temp_df["strat_key"], random_state=42)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    train_df.drop(columns="strat_key").to_csv(DATA_PATH, index=False)
    val_df.drop(columns="strat_key").to_csv(f"{DRIVE_ROOT}/data/processed/phishing_val.csv", index=False)
    test_df.drop(columns="strat_key").to_csv(f"{DRIVE_ROOT}/data/processed/phishing_test.csv", index=False)
    print("Successfully saved all 3 splits into your Google Drive permanently!")

df = pd.read_csv(DATA_PATH)
print(f"Loaded training dataset: {df.shape} ({df['label'].mean():.2%} phishing)")
if SMOKE_TEST:
    df = df.head(200).copy()
    print(f"SMOKE TEST ACTIVE: Subsampled to first {len(df)} rows.")


# CELL 6 — Inlined Prompt Template & Pre-format Truncation (Self-contained, no imports needed)
RESPONSE_TEMPLATE = "### Response:\n"

def truncate_body_text(body, max_tokens=MAX_BODY_TOKENS, tokenizer=None):
    if not body or not isinstance(body, str):
        return "", False
    tokens = tokenizer.encode(body, add_special_tokens=False)
    if len(tokens) > max_tokens:
        return tokenizer.decode(tokens[:max_tokens], skip_special_tokens=True), True
    return body, False

def build_prompt_pair(subject, body, label_int):
    s = str(subject) if pd.notna(subject) else ""
    b = str(body) if pd.notna(body) else ""
    prompt_str = f"### Instruction:\nClassify the following email as 'Phishing' or 'Legitimate'.\n\n### Email:\nSubject: {s}\n\nBody:\n{b}\n\n### Response:\n"
    label_str = "Phishing" if label_int == 1 else "Legitimate"
    return {"prompt": prompt_str, "completion": f" {label_str}"}

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Build prompt-completion dataset with pre-format truncation
truncated_count = 0
records = []
for _, row in df.iterrows():
    trunc_body, was_trunc = truncate_body_text(row['body'], max_tokens=MAX_BODY_TOKENS, tokenizer=tokenizer)
    if was_trunc:
        truncated_count += 1
    records.append(build_prompt_pair(row['subject'], trunc_body, int(row['label'])))

pct_truncated = (truncated_count / len(df)) * 100
print(f"Pre-format truncation: {truncated_count}/{len(df)} rows truncated ({pct_truncated:.2f}%)")
train_dataset = Dataset.from_pandas(pd.DataFrame(records))
print("Sample prompt:", repr(train_dataset[0]['prompt'][:80]), "...")
print("Sample completion:", repr(train_dataset[0]['completion']))


In [ ]:
# CELL 7 — Load model in 4-bit (QLoRA) with fp16 compute for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 if USE_FP16 else torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.use_cache = False


In [ ]:
# CELL 8 — Attach LoRA adapter
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# CELL 9 — Resumable Training with completion_only_loss=True
import time, json

# 1. Self-Verification of Loss Masking on 3 examples
print("\n--- [VERIFICATION] Prompt-Completion Loss Masking Check ---")
for i in range(min(3, len(train_dataset))):
    s = train_dataset[i]
    p_ids = tokenizer(s["prompt"], add_special_tokens=False)["input_ids"]
    c_ids = tokenizer(s["completion"], add_special_tokens=False)["input_ids"]
    if tokenizer.eos_token_id is not None:
        c_ids.append(tokenizer.eos_token_id)
    full_ids = p_ids + c_ids
    labels = [-100] * len(p_ids) + c_ids
    unmasked = [idx for idx, x in enumerate(labels) if x != -100]
    unmasked_tokens = [tokenizer.decode([full_ids[idx]]) for idx in unmasked]
    print(f"Example {i+1}: {len(full_ids)} tokens | {len(p_ids)} prompt tokens masked (-100) | Trained completion: {unmasked_tokens}")
    assert len(unmasked) == len(c_ids), "Masking assertion failed!"
print("--- [VERIFIED] Prompt tokens masked (-100); only completion tokens are trained! ---\n")

# 2. Check for Existing Checkpoints to Support Resumption
last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
if last_checkpoint is not None and not SMOKE_TEST:
    print(f"[RESUME]: Found previous checkpoint at {last_checkpoint}. Resuming training!")
else:
    last_checkpoint = None

max_steps = 40 if SMOKE_TEST else -1
epochs = 1 if SMOKE_TEST else NUM_EPOCHS
save_strategy = "no" if SMOKE_TEST else "steps"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=epochs,
    max_steps=max_steps,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5 if SMOKE_TEST else 20,
    save_strategy=save_strategy,
    save_steps=SAVE_STEPS if not SMOKE_TEST else 500,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    report_to="none",
    completion_only_loss=True,
    max_length=1024,
    seed=RANDOM_SEED
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset
)

t0 = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
wall_time = time.time() - t0
peak_mem = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
steps = trainer.state.global_step
steps_sec = steps / wall_time if wall_time > 0 else 0.0

print(f"\n--- Training Output Summary ---")
print(f"Wall time:       {wall_time:.2f}s")
print(f"Global steps:    {steps}")
print(f"Steps/sec:       {steps_sec:.3f}")
print(f"Peak GPU Memory: {peak_mem:.2f} GB")
print(f"Pct Truncated:   {pct_truncated:.2f}%")
if SMOKE_TEST:
    proj_sec = 3878 / steps_sec if steps_sec > 0 else 0
    print(f"Projected full run (3,878 steps): {proj_sec/3600:.2f} hours")


In [ ]:
# CELL 9 — Fine-tune with Completion-Only Loss (Prompt tokens masked from loss)
from trl import DataCollatorForCompletionOnlyLM

# HAZARD 4 FIX: Mask prompt tokens so loss is computed ONLY on completion/label
response_template = "### Response:\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=USE_FP16,
    bf16=USE_BF16,
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    data_collator=collator,
)

trainer.train()


In [ ]:
# CELL 10 — Save the LoRA adapter to Drive (small — a few MB, not the full model)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter to:", OUTPUT_DIR)
print("This is on Google Drive, so it survives a Colab disconnect.")


## Quick sanity-check inference

Run this after training to eyeball a couple of predictions before moving to the full evaluation harness (Week 8).


In [ ]:
# CELL 11 — Quick inference smoke test
model.eval()
test_email = "Subject: Your package could not be delivered\n\nBody: Click here within 12 hours to reschedule delivery or your package will be returned to sender."
prompt = f"### Instruction:\nClassify the following email as 'Phishing' or 'Legitimate'.\n\n### Email:\n{test_email}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))


---
### Reloading a saved adapter later (e.g. new Colab session, evaluation script)

```python
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)   # OUTPUT_DIR = your Drive path
```

### Next steps
- Replace the synthetic placeholder with your real `phishing_dataset.csv` from Week 2 — put it in Drive at `phishing-slm-research/data/processed/` so this notebook finds it automatically via `DRIVE_ROOT`.
- Run this notebook twice per model — `USE_COT = False` then `USE_COT = True` — to produce the naive and CoT-distilled arms for Table 3.
- For the CoT arm, first generate the `reasoning` column with a teacher model (Week 6's distillation step) — ask for that script when you're ready.
- Keep evaluation (accuracy/F1/AUROC) in a separate `eval/evaluate.py`, not this training notebook.
